<a href="https://colab.research.google.com/github/noone878/data-science-2026/blob/main/Pertemuan3_SEPTIAN_AL_RIZKI_230401010262.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [134]:
import pandas as pd
import numpy as np
from scipy.stats import mstats
import requests

In [135]:
# STEP -0 - Load & Ekspolrasi File
# Membaca file yang sudah diunggah ke folder "Files" di Google Colab
df = pd.read_csv('/content/3. housing_dirty.csv')

df.head()

,id,luas_m2,harga_juta,kota,kamar,tahun_bangun,kondisi
0,1,297.0,1084.0,jogja,2.0,2000,baik
1,2,254.0,761.0,Medan,NaN,1995,Bagus
2,3,249.7,895.0,Depok,NaN,1983,baik
3,4,49.7,178.0,YGY,5.0,2013,baik
4,5,133.4,424.0,Medan,5.0,2004,Sedang


In [136]:
# --- STEP 1: Pembersihan Duplikat ---
print("--- STEP 1: MENGHAPUS DUPLIKAT ---")
awal = len(df)
df.drop_duplicates(inplace=True)
akhir = len(df)
print(f"Data awal: {awal} baris | Data sekarang: {akhir} baris")
print(f"Berhasil menghapus {awal - akhir} baris duplikat.\n")

--- STEP 1: MENGHAPUS DUPLIKAT ---
Data awal: 130 baris | Data sekarang: 130 baris
Berhasil menghapus 0 baris duplikat.



In [137]:
#STEP  - Melihat Data
print("INFO")
df.info()

print("\nDESCRIBE")
display(df.describe())

print("\nMISSING VALUES")
display(df.isnull().sum())


INFO
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 130 entries, 0 to 129
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id            130 non-null    int64  
 1   luas_m2       112 non-null    float64
 2   harga_juta    113 non-null    float64
 3   kota          130 non-null    object 
 4   kamar         120 non-null    float64
 5   tahun_bangun  130 non-null    int64  
 6   kondisi       130 non-null    object 
dtypes: float64(3), int64(2), object(2)
memory usage: 7.2+ KB

DESCRIBE


,id,luas_m2,harga_juta,kamar,tahun_bangun
count,130.000000,112.000000,1.130000e+02,120.000000,130.000000
mean,65.500000,267.627679,8.856325e+05,3.433333,2062.638462
std,37.671829,885.664181,9.407144e+06,1.776283,701.684043
min,1.000000,-50.000000,-5.000000e+02,1.000000,1890.000000
25%,33.250000,87.050000,3.450000e+02,2.000000,1991.250000
50%,65.500000,193.800000,6.550000e+02,4.000000,2002.000000
75%,97.750000,280.675000,9.550000e+02,5.000000,2011.750000
max,130.000000,9500.000000,1.000000e+08,6.000000,9999.000000



MISSING VALUES


,0
id,0
luas_m2,18
harga_juta,17
kota,0
kamar,10
tahun_bangun,0
kondisi,0


In [138]:
# --- STEP 2: Normalisasi Nama Kota & Kondisi ---
city_mapping = {
    'Ygy': 'Yogyakarta', 'Jogja': 'Yogyakarta', 'Yogyakarta': 'Yogyakarta',
    'Sby': 'Surabaya', 'Smg': 'Semarang', 'Mksr': 'Makassar',
    'Mdn': 'Medan', 'Bdg': 'Bandung', 'Dpk': 'Depok'
}
df['kota'] = df['kota'].str.strip().str.title().replace(city_mapping)
df['kondisi'] = df['kondisi'].str.strip().str.lower()
print(f"Step 2: Normalisasi teks selesai.")

Step 2: Normalisasi teks selesai.


In [139]:
# --- STEP 3: Filter Logika (Ubah data mustahil menjadi NaN) ---
# Langkah ini harus dilakukan SEBELUM mengisi data kosong (Imputasi)
df.loc[df['luas_m2'] <= 0, 'luas_m2'] = np.nan
df.loc[df['tahun_bangun'] > 2026, 'tahun_bangun'] = np.nan
print(f"Step 3: Data tidak logis telah diubah menjadi NaN.")

Step 3: Data tidak logis telah diubah menjadi NaN.


In [140]:
# --- STEP 4: Imputasi (Mengisi Data Kosong) ---
# Sekarang kita isi SEMUA NaN, termasuk yang baru muncul dari Step 3
df['luas_m2'] = df['luas_m2'].fillna(df['luas_m2'].median())
df['harga_juta'] = df['harga_juta'].fillna(df['harga_juta'].median())
df['kamar'] = df['kamar'].fillna(df['kamar'].mode()[0])
print(f"Step 4: Semua data kosong (NaN) telah diisi.")

Step 4: Semua data kosong (NaN) telah diisi.


In [141]:
# --- STEP 5: Penanganan Outlier ---
for col in ['harga_juta', 'luas_m2']:
    Q1, Q3 = df[col].quantile([0.25, 0.75])
    IQR = Q3 - Q1
    df[col] = df[col].clip(lower=Q1 - 1.5*IQR, upper=Q3 + 1.5*IQR)
print(f"Step 5: Outlier telah dibatasi.")

Step 5: Outlier telah dibatasi.


In [142]:
# --- STEP 6: VALIDASI & EKSPOR ---
missing_total = df.isnull().sum().sum()
duplicate_total = df.duplicated().sum()

print("\n=== RINGKASAN VALIDASI AKHIR ===")
print(f"Total Data Kosong   : {missing_total}")
print(f"Total Data Duplikat : {duplicate_total}")
print(f"Ukuran Data Akhir   : {df.shape}")

# Validasi ketat menggunakan assert
assert missing_total == 0, f'Gagal! Masih ada {missing_total} data kosong.'
assert duplicate_total == 0, 'Gagal! Masih ada data duplikat.'


=== RINGKASAN VALIDASI AKHIR ===
Total Data Kosong   : 2
Total Data Duplikat : 0
Ukuran Data Akhir   : (130, 7)


AssertionError: Gagal! Masih ada 2 data kosong.